# 🚀 Fine-Tuning & Transfer Learning com Unsloth no Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanchagas04/personal-assistant/blob/feat/transfer_learning/notebooks/transfer_learning_unsloth.ipynb)

Este notebook realiza **Fine-Tuning real com ajuste de pesos (LoRA / QLoRA)** utilizando a biblioteca **Unsloth**, a ferramenta mais rápida e eficiente em consumo de memória para treinar modelos da família **LLaMA 3 / LLaMA 3.2**.

> ⚡ **Vantagens do Unsloth no Colab (GPU T4 Gratuita):**
> - Treinamento **2x a 5x mais rápido** com até 80% menos uso de VRAM.
> - Ajusta os pesos neurais do modelo para absorver com fidelidade o vocabulário, gírias e tom de Yan Chagas.
> - Exporta o modelo treinado diretamente para **GGUF**, permitindo importá-lo no seu **Ollama local**!

---

### 📌 Roteiro do Notebook:
1. **Instalação do Unsloth e Dependências** (otimizado para Colab GPU).
2. **Carregamento do Modelo Base LLaMA 3.2 em 4-bit** (`unsloth/Llama-3.2-3B-Instruct`).
3. **Configuração dos Adaptadores LoRA** (treinando apenas 1-2% dos parâmetros).
4. **Carregamento e Formatação do Dataset** (`chat_dataset_sample25.jsonl`).
5. **Treinamento com `SFTTrainer`** (~3 a 5 minutos na GPU T4).
6. **Inferência e Testes Comparativos** (validando o comportamento do clone).
7. **Exportação para GGUF & Importação no Ollama**.

## 1. Verificação de GPU e Instalação do Unsloth

In [ ]:
# Valida se o ambiente do Colab possui GPU ativa (T4, V100 ou A100)
!nvidia-smi

# Instalação do Unsloth e dependências compatíveis com Torch e CUDA do Colab
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets python-dotenv

## 2. Carregamento do Modelo Base em 4-bit

Utilizamos o **`Llama-3.2-3B-Instruct`** pré-quantizado em 4 bits da Unsloth (leve, rápido e excelente para diálogos em português).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None para detecção automática (Float16 ou Bfloat16)
load_in_4bit = True

# Modelo recomendado para clone digital no Colab
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

print(f"Carregando modelo {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Modelo base e Tokenizer carregados com sucesso!")

## 3. Configuração dos Adaptadores LoRA / QLoRA

Configuramos o LoRA para injetar matrizes de rank baixo nas camadas de atenção e MLP, permitindo que o modelo aprenda o novo estilo sem perder seu conhecimento prévio.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                # Rank do LoRA (16 ou 32 são ideais)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,      # Otimizado para 0 no Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ Adaptadores LoRA aplicados!")

## 4. Carregamento e Formatação do Dataset de Conversas

Clonamos o repositório para obter `data/processed/chat_dataset_sample25.jsonl` e aplicamos o template oficial de chat do LLaMA-3.

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Baixar o dataset do repositório caso não esteja presente
if not Path("personal-assistant").exists() and not Path("data").exists():
    !git clone -b feat/transfer_learning https://github.com/yanchagas04/personal-assistant.git
    data_path = Path("personal-assistant/data/processed/chat_dataset_sample25.jsonl")
elif Path("personal-assistant").exists():
    data_path = Path("personal-assistant/data/processed/chat_dataset_sample25.jsonl")
else:
    data_path = Path("data/processed/chat_dataset_sample25.jsonl")

with open(data_path, "r", encoding="utf-8") as f:
    raw_dialogues = [json.loads(line) for line in f]

print(f"Total de conversas carregadas: {len(raw_dialogues)}")

# Aplicar Chat Template do Llama-3.1 / Llama-3.2
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

hf_dataset = Dataset.from_list(raw_dialogues)
hf_dataset = hf_dataset.map(formatting_prompts_func, batched=True)

print("\n--- Exemplo formatado para treino ---")
print(hf_dataset[0]["text"][:400] + "...")

## 5. Treinamento Supervisionado com `SFTTrainer`

Iniciamos o ajuste fino dos adaptadores LoRA. Como temos 25 conversas de alta qualidade, o treino leva **menos de 3 minutos** no Colab.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Iniciando treinamento...")
trainer_stats = trainer.train()
print("🎉 Treinamento concluído com sucesso!")

## 6. Testes de Inferência do Clone Treinado

In [ ]:
# Prepara modelo para inferência acelerada
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "Você é o clone digital de Yan Chagas. Responde direto ao ponto e de forma autêntica."},
    {"role": "user", "content": "Bora almoçar onde hoje?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=0.7)
decoded = tokenizer.batch_decode(outputs)
print("🤖 Resposta gerada:")
print(decoded[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip())

## 7. Exportação do Modelo para GGUF (Pronto para Ollama!)

O Unsloth permite exportar o modelo treinado diretamente para o formato **GGUF** (quantização `q4_k_m`), que é o formato nativo do **Ollama**!

In [ ]:
# Salva e quantiza diretamente em formato GGUF
GGUF_DIR = "yan_clone_gguf"

print("Exportando modelo para GGUF (quantização q4_k_m)... aguarde alguns instantes...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print(f"✅ Modelo GGUF exportado em: {GGUF_DIR}")

# Listar arquivos exportados
!ls -lh {GGUF_DIR}

## 8. Como Usar o Modelo Treinado no seu Ollama Local

1. Baixe o arquivo `.gguf` gerado (ex: `yan_clone_gguf/Llama-3.2-3B-Instruct-Q4_K_M.gguf`).
2. No seu computador, crie um arquivo `Modelfile` apontando para o arquivo baixado:
   ```dockerfile
   FROM ./Llama-3.2-3B-Instruct-Q4_K_M.gguf
   PARAMETER temperature 0.7
   SYSTEM "Você é o clone digital de Yan Chagas. Responde direto ao ponto e de forma autêntica."
   ```
3. No terminal do seu computador, crie e rode o modelo:
   ```bash
   ollama create yan-clone-treinado -f Modelfile
   ollama run yan-clone-treinado
   ```